# Comparison of quantile delta mapping and detrended quantile mapping for bias correcting climate model dry-bulb temperature output. 

## Step 0: Set-up
Load in the libraries, and define the climakitae `app` object.

In [ ]:
import climakitae as ck
from climakitae.utils import get_closest_gridcell
import xarray as xr
from xclim.core.calendar import convert_calendar
from xclim.core.units import convert_units_to
from xclim.sdba.adjustment import QuantileDeltaMapping, DetrendedQuantileMapping
from xclim import sdba
import pkg_resources
import pandas as pd
from bokeh.models import HoverTool
from climakitae.utils import _read_ae_colormap

In [ ]:
app = ck.Application()

## Step 1: Read in the station data
Open the HadISD dataset for Sacramento Executive Airport (KSAC) temperatures and do some processing

In [ ]:
# Import stations names and coordinates file
stations = pkg_resources.resource_filename("climakitae", "data/hadisd_stations.csv")
stations_df = pd.read_csv(stations)
my_station = 'KSFO'
station_id = str(stations_df[stations_df['ID'] == my_station]['station id'].values[0])

filepaths = [
    "s3://cadcat/tmp/hadisd/HadISD_{}.zarr".format(s_id)
    for s_id in [station_id]
]

obs_ds = xr.open_mfdataset(
    filepaths,
    engine="zarr",
    consolidated=False,
    parallel=True,
    backend_kwargs=dict(storage_options={"anon": True}),
)

obs_ds = obs_ds.tas
obs_ds = convert_units_to(obs_ds, "degF")
obs_ds = convert_calendar(obs_ds, "noleap")
obs_ds = obs_ds.chunk(dict(
    time=-1)).compute()

# extract coordinates
lat0 = obs_ds.latitude.values
lon0 = obs_ds.longitude.values

## Step 2: Read in the model output
Here we specifically pick CESM2 because it is known to have a systematic warm bias.

In [ ]:
app.selections.scenario_historical=['Historical Climate']
app.selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
app.selections.append_historical = True
app.selections.area_average = 'No'
app.selections.time_slice = (1981, 2060)
app.selections.resolution = '3 km'
app.selections.timescale = 'hourly'
app.selections.variable = 'Air Temperature at 2m'
app.selections.units = 'degF'
app.selections.cached_area = 'coordinate selection'
app.selections.latitude = (lat0-1., lat0+1.)
app.selections.longitude = (lon0-1., lon0+1.)

wrf_ds = app.retrieve()
wrf_ds = wrf_ds.isel(simulation = 0).squeeze()

Extract the WRF grid cell closest to the station and process the data

In [ ]:
# convert calendar
wrf_ds = convert_calendar(wrf_ds, "noleap")
# extract closest grid cell
wrf_ds = get_closest_gridcell(wrf_ds,
                               lat0,lon0)
# need to unchunk for bias correction
wrf_ds = wrf_ds.chunk(dict(
    time=-1)).compute()
# do some renaming for plotting ease later
wrf_ds.attrs['physical_variable'] = wrf_ds.name
wrf_ds.name = 'Raw'

Define a rolling window (in days) over which to perform all calculations.

In [ ]:
window = 1 # days

## Step 3: Inspect the model data and observations
Here we show record-mean daily-mean temperatures for the observations and raw WRF data to get a sense of the bias in the WRF model.

In [ ]:
def group_ds(ds, obs_ds=obs_ds, projected_ceil=2060):
    
    proj_floor = str(projected_ceil-29)
    proj_ceil = str(projected_ceil)
    
    
    hist_ds = ds.sel(time=slice(str(obs_ds.time.values[0]),
            str(obs_ds.time.values[-1]))
                    ).rolling(time=window).mean().groupby(
            'time.dayofyear').mean()    
    ssp_ds = ds.sel(time=slice(proj_floor,proj_ceil)
                   ).rolling(time=window).mean().groupby(
            'time.dayofyear').mean()    
    obs_ds = obs_ds.rolling(time=window).mean().groupby(
        'time.dayofyear').mean()
    
    hist_ds = hist_ds.rename(dict(dayofyear = 'Day of Year'))
    ssp_ds = ssp_ds.rename(dict(dayofyear = 'Day of Year'))
    obs_ds = obs_ds.rename(dict(dayofyear = 'Day of Year'))
    
    return hist_ds, ssp_ds, obs_ds

def compare_raw_and_obs(ds, obs_ds=obs_ds, ylim=(None,None), 
                        width=700, height=300): 
    
    hist_gp, ssp_gp, obs_gp = group_ds(ds, obs_ds)
    
    hist_pl = hist_gp.hvplot(label='Historical raw',c='royalblue')    
    ssp_pl = ssp_gp.hvplot(label='Projected raw',c='goldenrod')    
    obs_pl = obs_gp.hvplot(label='Observations',c='k')
   
    pl = obs_pl * hist_pl * ssp_pl
    pl.opts(ylim=ylim, width=width, height=height, legend_position='right',
           toolbar='below', ylabel=obs_ds.units,title='Record-mean Daily-mean '
            +ds.attrs['physical_variable'])
    
    return pl

compare_raw_and_obs(wrf_ds, obs_ds=obs_ds)

## Step 4: Perform the bias correction procedure

In [ ]:
# quantile delta mapping
def do_QDM(obs, ds, nquantiles=20, 
           group='time.dayofyear', window=window, 
           kind="+"):
    
    group = sdba.Grouper(group, window=window)

    ds.attrs['variable'] = ds.name
    ds.name = 'Raw' 
    
    QDM = QuantileDeltaMapping.train(obs, ds.sel(
        time=slice(str(obs_ds.time.values[0]),
        str(obs_ds.time.values[-1]))), 
        nquantiles=nquantiles, group=group, kind=kind)
    
    ds_adj = QDM.adjust(ds).compute()
    
    QDM_ds = QDM.ds.rename(dict(
        dayofyear = 'Day of Year', 
        quantiles='Quantile'))    
    
    ds_adj.name = 'Adjusted' 
    ds_adj = xr.merge([ds, ds_adj])
    
    return QDM_ds,ds_adj

# detrended quantile mapping
def do_dEQM(obs, ds, nquantiles=20, 
            group='time.dayofyear', window=window,
            kind="+"):
    
    ds.attrs['variable'] = ds.name
    ds.name = 'Raw' 
    
    hist_ds = ds.sel(
        time=slice(str(obs_ds.time.values[0]),
        str(obs_ds.time.values[-1])))
    
    hist_group, _, obs_group = group_ds(ds, obs_ds)

    dEQM = DetrendedQuantileMapping.train(
        obs, hist_ds, nquantiles=nquantiles, 
        window=window, group=group, kind=kind)
    
    dEQM_ds_adj = dEQM.adjust(ds).compute()
    
    dEQM_ds = dEQM.ds.rename(dict(
        dayofyear = 'Day of Year', 
        quantiles='Quantile',
        hist_q = 'diff_hist_q',
    ))
        
    dEQM_ds['hist_q'] = (
        dEQM_ds['diff_hist_q'
               ] + hist_group)
    
    dEQM_ds_adj.name = 'Adjusted' 
    dEQM_ds_adj = xr.merge([ds, dEQM_ds_adj])
    
    return dEQM_ds, dEQM_ds_adj

# quantile mapping - leaving here for now in case anyone is interested
def do_EQM(obs, ds, nquantiles=20, 
            group='time.dayofyear', window=window,
            kind="+"):
    
    ds.attrs['variable'] = ds.name
    ds.name = 'Raw' 
    
    hist_ds = ds.sel(
        time=slice(str(obs_ds.time.values[0]),
        str(obs_ds.time.values[-1])))
    
    hist_group, _, obs_group = group_ds(ds, obs_ds)
    
    EQM = sdba.EmpiricalQuantileMapping.train(
        obs, hist_ds, nquantiles=nquantiles, 
        window=window, group=group, kind=kind)
    
    ds_adj = EQM.adjust(ds).compute()
    
    EQM_ds = EQM.ds.rename(dict(
        dayofyear = 'Day of Year', 
        quantiles='Quantile',
    ))
        
    
    ds_adj.name = 'Adjusted' 
    ds_adj = xr.merge([ds, ds_adj])
    
    return EQM_ds,ds_adj

In [ ]:
dEQM_adj_factors, dEQM_adj_ds = do_dEQM(obs_ds,wrf_ds)
QDM_adj_factors, QDM_adj_ds = do_QDM(obs_ds,wrf_ds)

diff_adjf_ds =  QDM_adj_factors['af'] - dEQM_adj_factors['af'] 
diff_adjf_hist = QDM_adj_factors['hist_q'] - dEQM_adj_factors['hist_q']
diff_adjf_ds = xr.merge([diff_adjf_ds, diff_adjf_hist])

## Step 5: Visualize the bias correction results
### Step 5a. Inspect the raw historical WRF quantiles and the adjustment factor for each quantile

In [ ]:
from bokeh.models import HoverTool
from climakitae.utils import _read_ae_colormap

raw_cmap = _read_ae_colormap(cmap='ae_orange', cmap_hex=True)
af_cmap = _read_ae_colormap(cmap='ae_diverging', cmap_hex=True)
    
hover_temp = HoverTool(description='Custom Tooltip', 
        tooltips=[('Quantile', '@Quantile'), 
        ('Day of Year', '@{Day_of_Year}'),
                 ('Temperature (degF)', '@hist_q')])
hover_adj = HoverTool(description='Custom Tooltip', 
        tooltips=[('Quantile', '@Quantile'), 
        ('Day of Year', '@{Day_of_Year}'),
                 ('Adjustment (degF)', '@af')])

def plot_af(adj_factors, width=550, height=350, 
            method='Delta quantile mapping',
           clim=(None,None), cmap=raw_cmap):

    raw_temp_qs = adj_factors['hist_q'].hvplot.quadmesh(
        x='Quantile',y='Day of Year',z='hist_q').opts(
        tools=[hover_temp], width=width, height=height,
        cmap=cmap, clabel="degF", xlabel="Historical "+
        "model quantiles", clim=clim)
    
    adjf_temp = adj_factors['af'].hvplot.quadmesh(
        x='Quantile',y='Day of Year',z='af').opts(
        tools=[hover_adj], width=width, height=height,
        cmap=af_cmap, clabel="degF", xlabel="Quantile "+
        "adjustment factors")
    
    pl = (raw_temp_qs.opts(
        title='%s' % (method)) + adjf_temp.opts(
        title='')).cols(2)

    return pl

raw_and_af = plot_af(
    QDM_adj_factors, method = 
    'Quantile delta mapping',
    clim=(40,75)) + plot_af(dEQM_adj_factors,
    method = 'Detrended quantile mapping',
    clim=(40,75)) + plot_af(diff_adjf_ds,
    method = 'Difference (Quantile delta - detrended quantile mapping)',
    clim=(-1,1), cmap=af_cmap)


raw_and_af.opts(
    title="Raw historical quantiles"
    + " and computed adjustment factors",
    toolbar='below').cols(2)

*I think that slight differences between historical model quantiles for the two methods are due to the detrending which takes place before computing them for the detrended quantile mapping approach.*

### Step 5b: Directly compare the relative changes in each quantile before and after adjustment

In [ ]:
def grouped_quantiles(da, quantiles):
    
    grouper = sdba.Grouper("time.dayofyear", window=window)    
    quants = grouper.apply(func='quantile',q=quantiles,da=da)
    quants = quants.rename(dict(
        dayofyear = 'Day of Year', 
        quantile='Quantile',
    ))    
    return quants

def compare_quantiles(adj_ds, quantiles,
            obs_ds=obs_ds, width=550, height=350, 
            method='Delta quantile mapping',
           clim=(None,None), cmap=raw_cmap,
            projected_ceil=2060):
    
    hover_delta = HoverTool(description='Custom Tooltip', 
        tooltips=[('Quantile', '@Quantile'), 
        ('Day of Year', '@{Day_of_Year}'),
        ('Change (degF)', '@Delta')])
    
    
    proj_floor = str(projected_ceil-29)
    proj_ceil = str(projected_ceil)
    
    adj_hist_ds = adj_ds.sel(
        time=slice(str(obs_ds.time.values[0]),
        str(obs_ds.time.values[-1])))       
    adj_ssp_ds = adj_ds.sel(time=slice(proj_floor,proj_ceil))

    
    hist_quants = grouped_quantiles(
        adj_hist_ds, quantiles)
    ssp_quants = grouped_quantiles(
        adj_ssp_ds, quantiles)
    
    raw_delta_quants = (ssp_quants.Raw 
                 - hist_quants.Raw)
                 # )/hist_quants.Raw)*100
    raw_delta_quants.name = 'Delta'
    adj_delta_quants = (ssp_quants.Adjusted 
                 - hist_quants.Adjusted)
                 # )/hist_quants.Adjusted)*100
    adj_delta_quants.name = 'Delta'
    
    raw_temp_qs = hist_quants.Raw.hvplot.quadmesh(
        x='Quantile',y='Day of Year').opts(
        width=width, height=height,
        cmap=cmap, clabel="degF", xlabel="Raw historical "+
        "model quantiles", clim=clim)
    raw_ssp_pl = ssp_quants.Raw.hvplot.quadmesh(
        x='Quantile',y='Day of Year').opts(
        width=width, height=height,
        cmap=cmap, clabel="degF", xlabel="Raw projected "+
        "model quantiles", clim=clim)    
    raw_delta_pl = raw_delta_quants.hvplot.quadmesh(
        x='Quantile',y='Day of Year').opts(
        tools=[hover_delta], width=width, height=height,
        cmap=af_cmap, clabel="degF", xlabel="Absolute change "+
        "in raw model quantiles", clim=(-15,15))
    
    raw_pl = (raw_temp_qs + raw_ssp_pl + raw_delta_pl).cols(1)
    
    adj_temp_qs = hist_quants.Adjusted.hvplot.quadmesh(
        x='Quantile',y='Day of Year').opts(
        width=width, height=height,
        cmap=cmap, clabel="degF", xlabel="Adjusted historical "+
        "model quantiles", clim=clim)
    adj_ssp_pl = ssp_quants.Adjusted.hvplot.quadmesh(
        x='Quantile',y='Day of Year').opts(
        width=width, height=height,
        cmap=cmap, clabel="degF", xlabel="Adjusted projected "+
        "model quantiles", clim=clim)    
    adj_delta_pl = adj_delta_quants.hvplot.quadmesh(
        x='Quantile',y='Day of Year').opts(
        tools=[hover_delta], width=width, height=height,
        cmap=af_cmap, clabel="degF", xlabel="Absolute change "+
        "in adjusted model quantiles", clim=(-15,15))
    
    adj_pl = (adj_temp_qs + adj_ssp_pl + adj_delta_pl).cols(1)
    
    hist_pl = (raw_temp_qs + adj_temp_qs).cols(2)
    ssp_pl = (raw_ssp_pl + adj_ssp_pl).cols(2)
    delta_pl = (raw_delta_pl + adj_delta_pl).cols(2)
    
    pl = (hist_pl + ssp_pl + delta_pl).cols(2)
#         #   .opts(
#         # title='%s' % (method)) + adjf_temp.opts(
#         # title='')).cols(1)

    return pl

quantiles = [round(q,3) for q in QDM_adj_factors.Quantile.values]
compare_quantiles(QDM_adj_ds, quantiles, clim=(30,85)
                 ).opts(title="Raw and adjusted quantiles and their absolute changes")

*I suppose QDM aims to preserve changes in quantiles, but isn't perfect. Still figuring this out.*
Here is the same plot as above, but for detrended quantile mapping.

In [ ]:
compare_quantiles(dEQM_adj_ds, quantiles, clim=(30,85)
                 ).opts(title="Raw and adjusted quantiles and their absolute changes")


### Step 5c. Some other plots to try to make sense of things

In [ ]:
def make_comparison_plot(hist_ds, ssp_ds, obs_ds=None, 
                         width=475, height=300,
                         ylim=(None,None), 
                         method='Delta quantile method'):
    
    y_str = hist_ds.physical_variable+' ('+hist_ds.attrs['units']+')'
    
    pl_historical = hist_ds.Raw.hvplot(
        color="royalblue", label='Historical '+hist_ds.Raw.name) 
    pl_historical *= hist_ds.Adjusted.hvplot(
        color="goldenrod", line_width=2,
        label='Historical '+hist_ds.Adjusted.name)    
    if obs_ds is not None:
        pl_historical *= obs_ds.hvplot(
            color='k',line_width=1, label="Observations")        
    pl_historical.opts(
        legend_position='top_left', width=width, 
        height=height, title='%s' % (method), 
        ylabel=y_str, ylim=ylim)
    
    pl_ssp = ssp_ds.Raw.hvplot(
        color="royalblue", label='Projected '+ssp_ds.Raw.name) 
    pl_ssp *= ssp_ds.Adjusted.hvplot(
        color="goldenrod", line_width=2,label='Projected ' 
        +ssp_ds.Adjusted.name)    
    pl_ssp.opts(
        legend_position='top_left', width=width, 
        height=height, ylabel=y_str,
        ylim=ylim, title="")
    
    return pl_historical + pl_ssp

#### 1. Record-mean daily-mean raw and adjusted data.

In [ ]:
hist_gp, ssp_gp, obs_gp = group_ds(QDM_adj_ds, obs_ds) 
QDM_pl = make_comparison_plot(hist_gp, ssp_gp, obs_ds=obs_gp
                    )
hist_gp, ssp_gp, obs_gp = group_ds(dEQM_adj_ds, obs_ds) 
dEQM_pl = make_comparison_plot(hist_gp, ssp_gp, obs_ds=obs_gp,
          method="Detrended quantile mapping"
                    )

(QDM_pl + dEQM_pl).opts(
    title="Record-mean Daily-mean "
    + "Raw and Adjusted Output "
    + "for Historical and Projected "
    + "Time Periods").cols(2)

*Okay, so the larger downward adjustment in the DQM historical makes sense if you look at the difference plot above. But why the opposite story in the projected data?*

#### 2. Let's see how the standard deviation is impacted by the two methods. Maybe that will give us some insight??

In [ ]:
def month_gp(ds, obs_ds=obs_ds, projected_ceil=2060):
    
    proj_floor = str(projected_ceil-29)
    proj_ceil = str(projected_ceil)    
    hist_ds = ds.sel(time=slice(str(
            obs_ds.time.values[0]),
            str(obs_ds.time.values[-1]))
            ).rolling(time=window).mean().groupby(
            'time.dayofyear').std()    
    ssp_ds = ds.sel(time=slice(proj_floor,
            proj_ceil)).rolling(time=window).mean(
    ).groupby('time.dayofyear').std()    
    obs_ds = obs_ds.rolling(time=window).mean().groupby('time.dayofyear').std()    
    return hist_ds, ssp_ds, obs_ds

mon_hist, mon_ssp, mon_obs = month_gp(QDM_adj_ds, obs_ds=obs_ds)
mon_QDM_std = make_comparison_plot(mon_hist, mon_ssp, obs_ds=mon_obs,
          ylim=(None,None))

mon_hist, mon_ssp, mon_obs = month_gp(dEQM_adj_ds, obs_ds=obs_ds)
mon_dEQM_std = make_comparison_plot(mon_hist, mon_ssp, obs_ds=mon_obs,
           ylim=(None,None), method = "Detrended quantile mapping")


(mon_QDM_std + mon_dEQM_std).cols(2)

*I cannot figure this out.*

#### 2. Annual mean data

In [ ]:
def ann_gp(ds, obs_ds=obs_ds, projected_ceil=2060):
    
    proj_floor = str(projected_ceil-29)
    proj_ceil = str(projected_ceil)    
    hist_ds = ds.sel(time=slice(str(
            obs_ds.time.values[0]),
            str(obs_ds.time.values[-1]))).groupby(
            'time.year').mean()    
    ssp_ds = ds.sel(time=slice(proj_floor,
            proj_ceil)).groupby('time.year').mean()    
    obs_ds = obs_ds.groupby('time.year').mean()    
    hist_ds = hist_ds.rename(dict(year = 'Year'))
    ssp_ds = ssp_ds.rename(dict(year = 'Year'))
    obs_ds = obs_ds.rename(dict(year = 'Year'))    
    return hist_ds, ssp_ds, obs_ds

ann_hist, ann_ssp, ann_obs = ann_gp(QDM_adj_ds, obs_ds=obs_ds)
ann_QDM = make_comparison_plot(ann_hist, ann_ssp, obs_ds=ann_obs,
          ylim=(52,64))

ann_hist, ann_ssp, ann_obs = ann_gp(dEQM_adj_ds, obs_ds=obs_ds)
ann_dEQM = make_comparison_plot(ann_hist, ann_ssp, obs_ds=ann_obs,
           ylim=(52,64), method = "Detrended quantile mapping")

(ann_QDM + ann_dEQM).cols(2).opts(
    title="Annual Mean Raw and Adjusted Output "
     + "for Historical and Projected Time Periods",
shared_axes=False)

#### 3. Zoom in on the hourly time series by sampling some weather extremes
Define a function to identify extremes in the timeseries.

In [ ]:
def sel_extremes(ds, obs_ds, projected_ceil=2060, window=72):
    
    window = window
    proj_floor = str(projected_ceil-29)
    proj_ceil = str(projected_ceil)    
    hist_ds = ds.sel(time=slice(str(
        obs_ds.time.values[0]),
        str(obs_ds.time.values[-1])))    
    hist_max = hist_ds.isel(time=slice(
        hist_ds.Raw.argmax().values-window,
        hist_ds.Raw.argmax().values+window))
    hist_min = hist_ds.isel(time=slice(
        hist_ds.Raw.argmin().values-window,
        hist_ds.Raw.argmin().values+window))
    
    ssp_ds = ds.sel(time=slice(proj_floor,
            proj_ceil))
    ssp_max = ssp_ds.isel(time=slice(
        ssp_ds.Raw.argmax().values-window,
        ssp_ds.Raw.argmax().values+window))
    ssp_min = ssp_ds.isel(time=slice(
        ssp_ds.Raw.argmin().values-window,
        ssp_ds.Raw.argmin().values+window))    

    return hist_min, hist_max, ssp_min, ssp_max
hist_min, hist_max, ssp_min, ssp_max = sel_extremes(QDM_adj_ds, obs_ds)

##### Plot record maxima for raw and adjusted historical and projected data

In [ ]:
make_comparison_plot(hist_max, ssp_max, ylim=(55, 100),
                    ).opts(shared_axes=False, 
                    title="Maximum hourly temperature +/- 72 hours")

##### Plot record minima for raw and adjusted historical and projected data

In [ ]:
make_comparison_plot(hist_min, ssp_min, ylim=(25, 60),
                    ).opts(shared_axes=False, 
                    title="Minimum hourly temperature +/- 72 hours")